# CS781 Phase 3 — Strong Teacher Search
## Phase-1 Model 4 + Rank-Stabilized LoRA (rsLoRA) — Rhetorical Role Prediction

**Goal:** maximize Phase-3 Macro-F1 before the mandatory Knowledge-Distillation experiment.

This notebook uses the exact Phase-1 **Model 4** checkpoint and applies **rsLoRA** to all seven useful projection matrices in every block:

- Attention: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- SwiGLU: `gate_proj`, `up_proj`, `down_proj`

### Why rsLoRA here?
Ordinary LoRA scales the low-rank update by `alpha / r`.  
Rank-Stabilized LoRA uses:

`scale = alpha / sqrt(r)`

which keeps higher-rank adapters from being excessively attenuated. We therefore use a deliberately larger rank (`r=32`) while keeping all original Model-4 weights frozen.

### Teacher-search configuration
- `r = 32`
- `alpha = 16`
- rsLoRA scaling = `16 / sqrt(32)` ≈ 2.83
- adapter dropout = `0.10`
- all 7 projections adapted
- context = ±2 sentences, `MAX_LEN=640`
- 4 dev evaluations per epoch to catch early peaks
- exact Phase-3 evaluator before refit
- fractional train+dev refit
- heldout / refit / ensemble submissions
- teacher logits exported for the later KD experiment

This is a distinct Phase-3 technique from plain LoRA and AdaLoRA.

> Attach the Phase-1 Model4 best checkpoint (`eff_full_best*.pt`) and the exact tokenizer, keep `HF_TOKEN` in Kaggle Secrets, use a T4, and run top-to-bottom.

### v2 changes
- **Scale/LR:** v1 used `alpha=32` → scaling 5.66 (≈2.8× plain LoRA's 2.0) together with a higher LR on 119 adapters: an aggressive effective step that Exp1 showed overfits within ~1 epoch. Now `alpha=16` (scaling 2.83), LR 1.5e-4, adapter dropout 0.10. rsLoRA's benefit (stable high rank) is kept.
- **Eval/accumulation alignment:** dev evaluations now fall on optimizer-update boundaries, so the fractional refit replays exactly the selected point.
- **New last code cell (24): cross-run teacher ensemble.** Averages probabilities from this run's heldout+refit logits and any other Model-4 runs' `test_logits.npz` (Exp1 LoRA, Exp5 AdaLoRA, …), applies the same transition decoding and writes `submission_teacher_ensemble.zip`. This can be useful as an optional post-hoc leaderboard ensemble. For KD, train-set teacher logits must be ensembled separately; test logits are never used to train the student.


### v3 additions
- keeps v2's safer `alpha=16`, dropout `0.10`, LR `1.5e-4`, and optimizer-aligned dev evaluation;
- saves the **top-3 heldout rsLoRA snapshots** by dev Macro-F1, not only the single best checkpoint;
- creates a **top-k snapshot ensemble** and a **snapshot+refit ensemble** on test, giving two extra strong rsLoRA-only submission candidates;
- cross-run test ensembling is now **safe-by-default**: it will not blindly average every `test_logits.npz` found on Kaggle;
- adds a separate, correctly aligned **train-set multi-teacher soft-target export** for KD when other Model-4 teacher-logit files are attached;
- fixes the report notes to match v2's actual `alpha=16`.


In [1]:
# ============================================================
# CELL 1 — Imports, seed, paths, and rsLoRA teacher-search config
# ============================================================
import os, re, gc, json, math, time, random, zipfile, subprocess, sys, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from huggingface_hub import HfApi, hf_hub_download, login

SEED = 1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Could not read Kaggle HF_TOKEN secret:", repr(e))
assert HF_TOKEN, "HF_TOKEN not found in Kaggle Secrets."
login(token=HF_TOKEN)
HF_REPO = "Exploration-Lab/CS-781-Capstone"

ROOT = Path("/kaggle/working/phase3_model4_rslora_teacher")
ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT = ROOT / "best_rslora_rr.pt"
FINAL_CKPT = ROOT / "final_refit_rslora_rr.pt"
DEV_PRED = ROOT / "prediction_dev.json"
CL_DEV_TRUTH = ROOT / "cl_dev_truth.json"
IT_DEV_TRUTH = ROOT / "it_dev_truth.json"
LABELS_JSON = ROOT / "labels.json"
EVAL_SCRIPT = ROOT / "evaluate_phase3.py"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# -------- Exact Phase-1 Model 4 --------
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 17
N_HEADS = 10
N_KV_HEADS = 2
FFN_DIM = 2048
MAX_POSITIONS = 2560
ROPE_BASE = 10_000.0
RMS_EPS = 1e-5
DROPOUT = 0.0
EXPECTED_TOKENIZER_SHA256 = "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

# -------- Phase-3 context --------
MAX_LEN = 640
CONTEXT_SENTS = 2
USE_POS = True
TARGET_BUDGET = 320
PREV_BUDGET = 120
NEXT_BUDGET = 120

# -------- rsLoRA --------
RSLORA_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]
RSLORA_R = 32
RSLORA_ALPHA = 16
RSLORA_DROPOUT = 0.10
RSLORA_LR = 1.5e-4
RSLORA_SCALING = RSLORA_ALPHA / math.sqrt(RSLORA_R)

# -------- RR task head --------
HEAD_DROPOUT = 0.15
HEAD_LR = 8e-4

# -------- Training --------
NUM_EPOCHS = 3
EVAL_PER_EPOCH = 4
EARLY_STOP_PATIENCE_EVALS = 6
LABEL_SMOOTHING = 0.03

# r=32 on all seven projections is larger than Exp1/Exp5 adapters;
# 4x4 keeps effective batch 16 while staying T4-safe.
MICRO_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4

WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.06
USE_AMP = True

CLASS_WEIGHT_MIN = 0.35
CLASS_WEIGHT_MAX = 3.50

PRINT_EVERY = 250
NUM_WORKERS = 2
LENGTH_GROUPED_TRAINING = True
LENGTH_BUCKET_MULT = 64
NAN_CHECK_EVERY = 50

REFIT_MATCH_ORIGINAL_SCHEDULE = True
ENSEMBLE_HELDOUT_AND_REFIT = True
TOPK_HELDOUT_SNAPSHOTS = 3
MAKE_SNAPSHOT_ENSEMBLES = True

print("Output directory:", ROOT)
print(
    f"Model4: {N_LAYERS} layers | d={D_MODEL} | "
    f"{N_HEADS}Q/{N_KV_HEADS}KV | RoPE | RMSNorm | SwiGLU"
)
print("rsLoRA targets:", RSLORA_TARGETS)
print(
    f"rsLoRA: r={RSLORA_R} alpha={RSLORA_ALPHA} "
    f"scale=alpha/sqrt(r)={RSLORA_SCALING:.6f} "
    f"dropout={RSLORA_DROPOUT}"
)
print("Context ±", CONTEXT_SENTS, "| MAX_LEN:", MAX_LEN)


Device: cuda
GPU: Tesla T4
Output directory: /kaggle/working/phase3_model4_rslora_teacher
Model4: 17 layers | d=640 | 10Q/2KV | RoPE | RMSNorm | SwiGLU
rsLoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
rsLoRA: r=32 alpha=16 scale=alpha/sqrt(r)=2.828427 dropout=0.1
Context ± 2 | MAX_LEN: 640


In [2]:
# ============================================================
# CELL 2 — Find Phase-1 Model 4 checkpoint and tokenizer safely
# ============================================================
def find_files(predicate):
    out=[]
    for base in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root, _, files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn)
                if predicate(fn,p): out.append(p)
    return out

ckpt_matches = find_files(
    lambda fn,p: fn.lower().endswith(".pt") and "eff_full_best" in fn.lower()
)
tok_matches = find_files(
    lambda fn,p: fn.lower().endswith(".json") and "tokenizer" in fn.lower()
)

print("Checkpoint candidates:")
for p in ckpt_matches: print(" ",p)
print("Tokenizer candidates:")
for p in tok_matches: print(" ",p)

assert ckpt_matches, "Attach the Phase-1 Model4 best checkpoint (eff_full_best*.pt)."
assert tok_matches, "Attach the exact Phase-1 tokenizer.json."

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):
            h.update(b)
    return h.hexdigest()

def score_path(p,words):
    low=p.lower()
    return sum(w in low for w in words)

PHASE1_CKPT=sorted(
    ckpt_matches,
    key=lambda p:(score_path(p,["model4","efficient","phase1","eff_full_best"]),p),
    reverse=True
)[0]

exact_tok=[p for p in tok_matches if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
TOKENIZER_FILE=exact_tok[0] if exact_tok else sorted(tok_matches)[0]

print("Using checkpoint:",PHASE1_CKPT)
print("Using tokenizer :",TOKENIZER_FILE)
print("Tokenizer SHA256:",sha256_file(TOKENIZER_FILE))

assert sha256_file(TOKENIZER_FILE)==EXPECTED_TOKENIZER_SHA256, (
    "Tokenizer hash mismatch — attach the exact Phase-1 tokenizer."
)


Checkpoint candidates:
  /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Tokenizer candidates:
  /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Using checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Using tokenizer : /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# ============================================================
# CELL 3 — Original custom byte-level LegalBPE
# ============================================================

BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            tok.symbol_bytes[s] = bytes([b])
        for l, r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:", len(tokenizer))
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)

probe = "IN THE SUPREME COURT OF INDIA\nSection 14: legal test."
assert tokenizer.decode(tokenizer.encode(probe)) == probe
print("✅ tokenizer round-trip passed")

Tokenizer vocab: 24576
✅ tokenizer round-trip passed


In [4]:
# ============================================================
# CELL 4 — Exact Phase-1 Model 4 architecture
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return x32.to(dtype) * self.weight


class RotaryTable(nn.Module):
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
        self.max_positions = max_positions

    def forward(self, position_ids):
        return (
            self.cos_cached[position_ids].unsqueeze(1),
            self.sin_cached[position_ids].unsqueeze(1),
        )


def apply_rope(x, cos, sin):
    B, H, T, D = x.shape
    cos = cos.to(x.dtype)
    sin = sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack(
        (x0 * cos - x1 * sin, x0 * sin + x1 * cos),
        dim=-1
    ).flatten(-2)


class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH = True

    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        new_kv = (k, v) if use_cache else None
        S = k.size(2)
        drop = self.dropout if self.training else 0.0

        if T == 1 and past_len > 0 and self.GQA_DECODE_PATH:
            qg = q.reshape(B, KV, self.kv_repeat, D)
            mask = None if key_valid is None else key_valid[:, None, None, :]
            y = F.scaled_dot_product_attention(
                qg, k, v, attn_mask=mask, dropout_p=drop, is_causal=False
            )
            y = y.reshape(B, H, 1, D).transpose(1, 2).contiguous().view(B, 1, C)
            return self.o_proj(y), new_kv

        if self.kv_repeat != 1:
            k_attn = k.repeat_interleave(self.kv_repeat, dim=1)
            v_attn = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            k_attn, v_attn = k, v

        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]

        y = F.scaled_dot_product_attention(
            q, k_attn, v_attn,
            attn_mask=attn_mask,
            dropout_p=drop,
            is_causal=is_causal
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.norm1(x), cos, sin,
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class EfficientDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
        ffn_dim, dropout, max_positions, rope_base, rms_eps
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(d_model // n_heads, max_positions, rope_base)

        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(
                d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps
            )
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model, rms_eps)

        self.apply(self._init_weights)
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff.down_proj.weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self, input_ids, past_key_values=None, use_cache=False,
        position_ids=None, key_valid=None
    ):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        cos, sin = self.rope(position_ids)

        x = self.drop(self.token_emb(input_ids))
        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x, cos, sin,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid
            )
            if use_cache:
                new_cache.append(kv)

        return self.final_norm(x), new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_backbone():
    return EfficientDecoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_positions=MAX_POSITIONS,
        rope_base=ROPE_BASE,
        rms_eps=RMS_EPS,
    )

backbone = build_backbone()
print("Fresh backbone params:", f"{sum(p.numel() for p in backbone.parameters())/1e6:.3f}M")

Fresh backbone params: 99.309M


In [5]:
# ============================================================
# CELL 5 — Load Phase-1 Model 4 checkpoint exactly
# ============================================================

phase1_obj = torch.load(PHASE1_CKPT, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(phase1_obj.keys()))

state = phase1_obj["model"] if isinstance(phase1_obj, dict) and "model" in phase1_obj else phase1_obj

missing, unexpected = backbone.load_state_dict(state, strict=True)
assert not missing and not unexpected

print("✅ Phase-1 Model 4 weights loaded with strict=True")
print("global_step:", phase1_obj.get("global_step") if isinstance(phase1_obj, dict) else None)
print("val_loss:", phase1_obj.get("val_loss") if isinstance(phase1_obj, dict) else None)
del state
gc.collect()

Checkpoint keys: ['model', 'global_step', 'val_loss', 'config']
✅ Phase-1 Model 4 weights loaded with strict=True
global_step: 16266
val_loss: 1.3664915735571412


63

In [6]:
# ============================================================
# CELL 6 — Custom rsLoRA from scratch
# ============================================================
class RsLoRALinear(nn.Module):
    """
    Frozen base linear layer + rank-stabilized low-rank update.

        y = W0 x + (alpha / sqrt(r)) * B(A(dropout(x)))

    B starts at zero, so injection is function-preserving at step 0.
    """
    def __init__(self, base_linear, r=RSLORA_R, alpha=RSLORA_ALPHA, dropout=RSLORA_DROPOUT):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)
        assert r > 0

        self.base = base_linear
        self.r = int(r)
        self.alpha = float(alpha)
        self.scaling = self.alpha / math.sqrt(self.r)
        self.rs_dropout = nn.Dropout(dropout)

        self.rslora_A = nn.Parameter(
            torch.empty(self.r, base_linear.in_features)
        )
        self.rslora_B = nn.Parameter(
            torch.zeros(base_linear.out_features, self.r)
        )

        nn.init.kaiming_uniform_(self.rslora_A, a=math.sqrt(5))
        nn.init.zeros_(self.rslora_B)

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        base_out = self.base(x)
        delta = F.linear(
            F.linear(self.rs_dropout(x), self.rslora_A),
            self.rslora_B,
        )
        return base_out + self.scaling * delta


def freeze_all(module):
    for p in module.parameters():
        p.requires_grad = False


def inject_rslora(
    model,
    targets=RSLORA_TARGETS,
    r=RSLORA_R,
    alpha=RSLORA_ALPHA,
    dropout=RSLORA_DROPOUT,
):
    freeze_all(model)
    replaced = []

    for i, block in enumerate(model.blocks):
        groups = (
            ("attn", block.attn, ["q_proj","k_proj","v_proj","o_proj"]),
            ("ff", block.ff, ["gate_proj","up_proj","down_proj"]),
        )
        for group_name, mod, names in groups:
            for name in names:
                if name in targets:
                    old = getattr(mod, name)
                    setattr(
                        mod,
                        name,
                        RsLoRALinear(
                            old, r=r, alpha=alpha, dropout=dropout
                        ),
                    )
                    replaced.append(f"blocks.{i}.{group_name}.{name}")

    return replaced


replaced = inject_rslora(backbone)

EXPECTED_MODULES = N_LAYERS * len(RSLORA_TARGETS)
assert len(replaced) == EXPECTED_MODULES, (len(replaced), EXPECTED_MODULES)

# Exact trainable-adapter count.
rslora_tensors = [
    (n,p) for n,p in backbone.named_parameters()
    if p.requires_grad and ("rslora_A" in n or "rslora_B" in n)
]
assert len(rslora_tensors) == 2 * EXPECTED_MODULES

# Function-preserving initialization check: all B matrices must be exactly zero.
assert all(
    torch.count_nonzero(p.detach()).item() == 0
    for n,p in rslora_tensors if "rslora_B" in n
)

rslora_params = sum(p.numel() for _,p in rslora_tensors)
backbone_total = sum(p.numel() for p in backbone.parameters())

print("Injected rsLoRA modules:", len(replaced))
print("Expected modules:", EXPECTED_MODULES)
print("First targets:", replaced[:10])
print("rsLoRA trainable params:", f"{rslora_params/1e6:.3f}M")
print("Backbone total params:", f"{backbone_total/1e6:.3f}M")
print("Adapter trainable %:", f"{100*rslora_params/backbone_total:.4f}%")
print("Scaling:", RSLORA_SCALING)

# GQA sanity: K/V projections are smaller, but still valid rsLoRA targets.
print(
    "Projection shapes:",
    "Q", tuple(backbone.blocks[0].attn.q_proj.base.weight.shape),
    "K", tuple(backbone.blocks[0].attn.k_proj.base.weight.shape),
    "V", tuple(backbone.blocks[0].attn.v_proj.base.weight.shape),
    "O", tuple(backbone.blocks[0].attn.o_proj.base.weight.shape),
)


Injected rsLoRA modules: 119
Expected modules: 119
First targets: ['blocks.0.attn.q_proj', 'blocks.0.attn.k_proj', 'blocks.0.attn.v_proj', 'blocks.0.attn.o_proj', 'blocks.0.ff.gate_proj', 'blocks.0.ff.up_proj', 'blocks.0.ff.down_proj', 'blocks.1.attn.q_proj', 'blocks.1.attn.k_proj', 'blocks.1.attn.v_proj']
rsLoRA trainable params: 6.615M
Backbone total params: 105.924M
Adapter trainable %: 6.2451%
Scaling: 2.82842712474619
Projection shapes: Q (640, 640) K (128, 640) V (128, 640) O (640, 640)


In [7]:
# ============================================================
# CELL 7 — Download Phase-3 raw JSON files (bypass datasets CastError)
# ============================================================

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)

def exact_repo_file(filename):
    matches = [
        f for f in repo_files
        if os.path.basename(f).lower() == filename.lower()
    ]
    assert len(matches) == 1, (filename, matches)
    return matches[0]

FILES = {
    "CL_train": exact_repo_file("CL_train.json"),
    "CL_dev": exact_repo_file("CL_dev.json"),
    "IT_train": exact_repo_file("IT_train.json"),
    "IT_dev": exact_repo_file("IT_dev.json"),
    "CL_test": exact_repo_file("CL_test.json"),
    "IT_test": exact_repo_file("IT_test.json"),
    "label_vocab": exact_repo_file("label_vocab.json"),
}

print(json.dumps(FILES, indent=2))

def load_repo_json(repo_path):
    p = hf_hub_download(
        repo_id=HF_REPO,
        filename=repo_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

cl_train_raw = load_repo_json(FILES["CL_train"])
cl_dev_raw   = load_repo_json(FILES["CL_dev"])
it_train_raw = load_repo_json(FILES["IT_train"])
it_dev_raw   = load_repo_json(FILES["IT_dev"])
cl_test_raw  = load_repo_json(FILES["CL_test"])
it_test_raw  = load_repo_json(FILES["IT_test"])
label_vocab_raw = load_repo_json(FILES["label_vocab"])

assert isinstance(cl_train_raw, list) and isinstance(it_train_raw, list)
assert isinstance(cl_dev_raw, list) and isinstance(it_dev_raw, list)
assert isinstance(cl_test_raw, dict) and isinstance(it_test_raw, dict)

print("CL train/dev:", len(cl_train_raw), len(cl_dev_raw))
print("IT train/dev:", len(it_train_raw), len(it_dev_raw))
print("CL/IT test docs:", len(cl_test_raw), len(it_test_raw))

{
  "CL_train": "Phase_3_RR/Train/CL_train.json",
  "CL_dev": "Phase_3_RR/Train/CL_dev.json",
  "IT_train": "Phase_3_RR/Train/IT_train.json",
  "IT_dev": "Phase_3_RR/Train/IT_dev.json",
  "CL_test": "Phase_3_RR/Test/CL_test.json",
  "IT_test": "Phase_3_RR/Test/IT_test.json",
  "label_vocab": "Phase_3_RR/Train/label_vocab.json"
}


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL train/dev: 40 5
IT train/dev: 40 5
CL/IT test docs: 5 5


In [8]:
# ============================================================
# CELL 8 — Labels, examples, class weights, and data checks
# ============================================================

def parse_label_vocab(data):
    if isinstance(data, list):
        labels = list(data)
        label_to_id = {x:i for i,x in enumerate(labels)}
    elif isinstance(data, dict):
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            label_to_id = {str(k): int(v) for k, v in data.items()}
            assert sorted(label_to_id.values()) == list(range(len(label_to_id)))
            labels = [None] * len(label_to_id)
            for lab, idx in label_to_id.items():
                labels[idx] = lab
        elif all(isinstance(v, str) for v in vals):
            tmp = {int(k): v for k, v in data.items()}
            labels = [tmp[i] for i in sorted(tmp)]
            label_to_id = {x:i for i,x in enumerate(labels)}
        else:
            raise ValueError("Unsupported label_vocab format")
    else:
        raise ValueError("Unsupported label_vocab format")

    return labels, label_to_id

LABELS, LABEL_TO_ID = parse_label_vocab(label_vocab_raw)
ID_TO_LABEL = {v:k for k,v in LABEL_TO_ID.items()}
NUM_LABELS = len(LABELS)

assert NUM_LABELS == 13
print("Labels:")
for i, x in enumerate(LABELS):
    print(i, x)

def normalize_id(doc_id):
    doc_id = str(doc_id)
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id

def validate_labeled_docs(docs, name):
    ids = set()
    n_sent = 0
    for d in docs:
        did = normalize_id(d["id"])
        assert did not in ids, f"{name}: duplicate ID {did}"
        ids.add(did)
        assert isinstance(d["text"], list)
        assert isinstance(d["labels"], list)
        assert len(d["text"]) == len(d["labels"]), did
        assert all(x in LABEL_TO_ID for x in d["labels"]), did
        n_sent += len(d["text"])
    print(f"{name}: docs={len(docs)} sentences={n_sent}")
    return ids

cltr_ids = validate_labeled_docs(cl_train_raw, "CL_train")
cldv_ids = validate_labeled_docs(cl_dev_raw, "CL_dev")
ittr_ids = validate_labeled_docs(it_train_raw, "IT_train")
itdv_ids = validate_labeled_docs(it_dev_raw, "IT_dev")

assert not (cltr_ids & cldv_ids)
assert not (ittr_ids & itdv_ids)
assert not (cldv_ids & itdv_ids)

def position_features(i, n):
    if not USE_POS:
        return [0.0, 0.0, 0.0, 0.0]
    return [i / max(1, n - 1), math.log1p(n) / 8.0, float(i == 0), float(i == n - 1)]

# Flatten into sentence-level records while preserving document context.
def make_labeled_records(docs, domain):
    out = []
    for d in docs:
        sents = d["text"]
        labs = d["labels"]
        did = normalize_id(d["id"])
        for i, (sent, lab) in enumerate(zip(sents, labs)):
            out.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
                "label": lab,
                "label_id": LABEL_TO_ID[lab],
            })
    return out

train_records = make_labeled_records(cl_train_raw, "CL") + make_labeled_records(it_train_raw, "IT")
dev_records   = make_labeled_records(cl_dev_raw, "CL") + make_labeled_records(it_dev_raw, "IT")

print("Train sentence records:", len(train_records))
print("Dev sentence records  :", len(dev_records))

train_counts = Counter(x["label"] for x in train_records)
print("\nTrain distribution:")
for lab in LABELS:
    print(f"{lab:30s} {train_counts[lab]:6d}")

# sqrt inverse-frequency; normalize mean to 1; then cap extremes.
N = len(train_records)
C = NUM_LABELS
raw_w = np.array(
    [math.sqrt(N / (C * max(train_counts[lab], 1))) for lab in LABELS],
    dtype=np.float32
)
raw_w = raw_w / raw_w.mean()
class_w = np.clip(raw_w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
class_w = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)

print("\nClass weights:")
for lab, w in zip(LABELS, class_w.detach().cpu().tolist()):
    print(f"{lab:30s} {w:.4f}")

Labels:
0 Fact
1 Issue
2 ArgumentPetitioner
3 ArgumentRespondent
4 Statute
5 Dissent
6 PrecedentReliedUpon
7 PrecedentNotReliedUpon
8 PrecedentOverruled
9 RulingByLowerCourt
10 RatioOfTheDecision
11 RulingByPresentCourt
12 None
CL_train: docs=40 sentences=10593
CL_dev: docs=5 sentences=1396
IT_train: docs=40 sentences=6252
IT_dev: docs=5 sentences=746
Train sentence records: 16845
Dev sentence records  : 2142

Train distribution:
Fact                             4098
Issue                             107
ArgumentPetitioner               1613
ArgumentRespondent               1348
Statute                          1099
Dissent                           284
PrecedentReliedUpon              1945
PrecedentNotReliedUpon            569
PrecedentOverruled                  3
RulingByLowerCourt                874
RatioOfTheDecision               4301
RulingByPresentCourt              307
None                              297

Class weights:
Fact                           0.3500
Issue             

In [9]:
# ============================================================
# CELL 9 — Context construction and tokenization
# ============================================================

_ENC_CACHE = {}
def enc(text):
    hit = _ENC_CACHE.get(text)
    if hit is None:
        hit = tokenizer.encode(text)
        _ENC_CACHE[text] = hit
    return hit

def crop_head_tail(ids, budget):
    if len(ids) <= budget:
        return ids
    if budget <= 0:
        return []
    left = budget // 2
    right = budget - left
    return ids[:left] + ids[-right:]

def build_context_ids(domain, prev_text, target_text, next_text, max_len=MAX_LEN):
    # Ordinary text markers only: no tokenizer-vocabulary modification.
    prefix = enc(f"Domain: {domain}\n")
    pmark  = enc("Previous sentence:\n")
    tmark  = enc("\nTarget sentence:\n")
    nmark  = enc("\nNext sentence:\n")
    rmark  = enc("\nRhetorical role:")

    prev_ids = enc(prev_text) if prev_text else []
    targ_ids = enc(target_text)
    next_ids = enc(next_text) if next_text else []

    # Initial caps protect the target sentence and preserve both neighboring sides.
    prev_ids = prev_ids[-PREV_BUDGET:]
    targ_ids = crop_head_tail(targ_ids, TARGET_BUDGET)
    next_ids = next_ids[:NEXT_BUDGET]

    fixed = prefix + pmark + tmark + nmark + rmark
    remaining = max_len - len(fixed)

    if remaining < 8:
        raise RuntimeError("Marker text unexpectedly consumes MAX_LEN.")

    # Fit segments to remaining space, with target priority.
    t_keep = min(len(targ_ids), max(8, min(TARGET_BUDGET, remaining)))
    remaining2 = remaining - t_keep

    p_keep = min(len(prev_ids), remaining2 // 2)
    n_keep = min(len(next_ids), remaining2 - p_keep)

    # Give any spare capacity back to previous/next, then target if available.
    spare = remaining2 - p_keep - n_keep
    if spare > 0:
        add_p = min(spare, len(prev_ids) - p_keep)
        p_keep += add_p
        spare -= add_p
    if spare > 0:
        add_n = min(spare, len(next_ids) - n_keep)
        n_keep += add_n
        spare -= add_n

    prev_ids = prev_ids[-p_keep:] if p_keep else []
    targ_ids = crop_head_tail(targ_ids, t_keep)
    next_ids = next_ids[:n_keep] if n_keep else []

    ids = (
        prefix
        + pmark + prev_ids
        + tmark + targ_ids
        + nmark + next_ids
        + rmark
    )

    if len(ids) > max_len:
        # Absolute safety fallback; preserve the terminal role marker.
        ids = ids[-max_len:]

    assert 1 <= len(ids) <= max_len
    return ids

# Token-length audit
lengths = [
    len(build_context_ids(
        r["domain"], r["prev"], r["target"], r["next"]
    ))
    for r in (train_records[:1000] + dev_records[:500])
]

print("Context-token sample:")
print(" mean:", round(float(np.mean(lengths)), 2))
print(" p95 :", round(float(np.percentile(lengths, 95)), 2))
print(" max :", max(lengths))

Context-token sample:
 mean: 317.94
 p95 : 404.0
 max : 594


In [10]:
# ============================================================
# CELL 10 — Dataset, collator, RR model (Model4 + rsLoRA)
# ============================================================

class RRDataset(Dataset):
    def __init__(self, records, labeled=True):
        self.records = records
        self.labeled = labeled

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        ids = build_context_ids(
            r["domain"], r["prev"], r["target"], r["next"]
        )
        item = {
            "input_ids": ids,
            "doc_id": r["doc_id"],
            "sent_idx": r["sent_idx"],
            "domain": r["domain"],
            "pos": r["pos"],
        }
        if self.labeled:
            item["label_id"] = r["label_id"]
        return item


def collate_rr(batch):
    max_t = max(len(x["input_ids"]) for x in batch)
    B = len(batch)

    input_ids = torch.full(
        (B, max_t),
        tokenizer.pad_token_id,
        dtype=torch.long
    )
    key_valid = torch.zeros((B, max_t), dtype=torch.bool)
    lengths = torch.zeros(B, dtype=torch.long)
    pos = torch.tensor([x["pos"] for x in batch], dtype=torch.float32)

    for i, x in enumerate(batch):
        ids = torch.tensor(x["input_ids"], dtype=torch.long)
        L = len(ids)
        input_ids[i, :L] = ids
        key_valid[i, :L] = True
        lengths[i] = L

    out = {
        "input_ids": input_ids,
        "key_valid": key_valid,
        "lengths": lengths,
        "pos": pos,
        "doc_id": [x["doc_id"] for x in batch],
        "sent_idx": [x["sent_idx"] for x in batch],
        "domain": [x["domain"] for x in batch],
    }

    if "label_id" in batch[0]:
        out["labels"] = torch.tensor(
            [x["label_id"] for x in batch],
            dtype=torch.long
        )

    return out


HEAD_PREFIXES = ("head_norm.", "pos_mlp.", "fc1.", "classifier.")

class RRClassifier(nn.Module):
    def __init__(self, backbone, num_labels=13, dropout=0.15):
        super().__init__()
        self.backbone = backbone
        self.head_norm = RMSNorm(2 * D_MODEL, RMS_EPS)
        self.pos_mlp = nn.Sequential(nn.Linear(4, 64), nn.GELU())
        self.head_drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(2 * D_MODEL + 64, 512)
        self.classifier = nn.Linear(512, num_labels)

    def forward(self, input_ids, key_valid, lengths, pos):
        h, _ = self.backbone.forward_hidden(
            input_ids, key_valid=key_valid, use_cache=False
        )
        h = h.float()
        row = torch.arange(h.size(0), device=h.device)
        last_h = h[row, lengths - 1]                      # sees whole context (causal) after the role marker
        m = key_valid.unsqueeze(-1).to(h.dtype)
        mean_h = (h * m).sum(1) / m.sum(1).clamp(min=1)
        x = self.head_norm(torch.cat([last_h, mean_h], dim=-1))
        z = torch.cat([x, self.pos_mlp(pos.float())], dim=-1)
        z = F.gelu(self.fc1(self.head_drop(z)))
        return self.classifier(self.head_drop(z))


model = RRClassifier(backbone, NUM_LABELS, HEAD_DROPOUT).to(DEVICE)

# Classifier is trainable; Phase-1 backbone is frozen except rsLoRA A/B.
for n_, p in model.named_parameters():
    if n_.startswith(HEAD_PREFIXES):
        p.requires_grad = True

# Strictly forbid accidental Phase-1 weight updates.
for n_, p in model.named_parameters():
    if p.requires_grad:
        assert (
            "rslora_A" in n_ or "rslora_B" in n_ or n_.startswith(HEAD_PREFIXES)
        ), f"Unexpected trainable parameter: {n_}"

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(x[1] for x in trainable)
n_total = sum(p.numel() for p in model.parameters())

print("Total params    :", f"{n_total/1e6:.3f}M")
print("Trainable params:", f"{n_train/1e6:.3f}M")
print("Trainable %     :", f"{100*n_train/n_total:.4f}%")
print("Trainable tensors:", len(trainable))
print("First trainable names:")
for x in trainable[:12]:
    print(" ", x)

Total params    : 106.621M
Trainable params: 7.312M
Trainable %     : 6.8579%
Trainable tensors: 245
First trainable names:
  ('backbone.blocks.0.attn.q_proj.rslora_A', 20480)
  ('backbone.blocks.0.attn.q_proj.rslora_B', 20480)
  ('backbone.blocks.0.attn.k_proj.rslora_A', 20480)
  ('backbone.blocks.0.attn.k_proj.rslora_B', 4096)
  ('backbone.blocks.0.attn.v_proj.rslora_A', 20480)
  ('backbone.blocks.0.attn.v_proj.rslora_B', 4096)
  ('backbone.blocks.0.attn.o_proj.rslora_A', 20480)
  ('backbone.blocks.0.attn.o_proj.rslora_B', 20480)
  ('backbone.blocks.0.ff.gate_proj.rslora_A', 20480)
  ('backbone.blocks.0.ff.gate_proj.rslora_B', 65536)
  ('backbone.blocks.0.ff.up_proj.rslora_A', 20480)
  ('backbone.blocks.0.ff.up_proj.rslora_B', 65536)


In [11]:
# ============================================================
# CELL 11 — Dataloaders, optimizer, scheduler
# ============================================================
class SortishBatchSampler(torch.utils.data.Sampler):
    """Deterministic shuffled length buckets to reduce padding."""
    def __init__(self, lengths, batch_size, seed=SEED, bucket_mult=LENGTH_BUCKET_MULT):
        self.lengths = list(map(int,lengths))
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.bucket_size = max(
            self.batch_size,
            self.batch_size * int(bucket_mult)
        )
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return math.ceil(len(self.lengths)/self.batch_size)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        idx = np.arange(len(self.lengths), dtype=np.int64)
        rng.shuffle(idx)
        batches = []
        for s in range(0, len(idx), self.bucket_size):
            chunk = idx[s:s+self.bucket_size]
            chunk = sorted(
                chunk.tolist(),
                key=lambda i:self.lengths[i],
                reverse=True
            )
            batches.extend(
                chunk[j:j+self.batch_size]
                for j in range(0,len(chunk),self.batch_size)
            )
        rng.shuffle(batches)
        yield from batches


train_ds = RRDataset(train_records, labeled=True)
dev_ds = RRDataset(dev_records, labeled=True)

# One-time lengths for efficient sortish batching.
train_lengths = [
    len(build_context_ids(r["domain"], r["prev"], r["target"], r["next"]))
    for r in train_records
]

if LENGTH_GROUPED_TRAINING:
    train_loader = DataLoader(
        train_ds,
        batch_sampler=SortishBatchSampler(
            train_lengths, MICRO_BATCH_SIZE, seed=SEED
        ),
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=NUM_WORKERS>0,
    )
else:
    g=torch.Generator(); g.manual_seed(SEED)
    train_loader=DataLoader(
        train_ds,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        generator=g,
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=NUM_WORKERS>0,
    )

dev_loader=DataLoader(
    dev_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type=="cuda"),
    persistent_workers=NUM_WORKERS>0,
)

adapter_params=[]
head_params=[]
for n,p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "rslora_A" in n or "rslora_B" in n:
        adapter_params.append(p)
    else:
        assert n.startswith(HEAD_PREFIXES),n
        head_params.append(p)

assert adapter_params and head_params

groups=[
    {"params":adapter_params, "lr":RSLORA_LR, "weight_decay":0.0},
    {"params":head_params, "lr":HEAD_LR, "weight_decay":WEIGHT_DECAY},
]
if DEVICE.type=="cuda":
    try:
        optimizer=torch.optim.AdamW(groups, fused=True)
        print("AdamW fused: True")
    except Exception as e:
        print("Fused AdamW unavailable:",repr(e)[:120])
        optimizer=torch.optim.AdamW(groups)
else:
    optimizer=torch.optim.AdamW(groups)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup_updates=max(1,int(total_updates*WARMUP_RATIO))

def lr_mult(step):
    if step<warmup_updates:
        return max(1e-8,step/warmup_updates)
    p=(step-warmup_updates)/max(1,total_updates-warmup_updates)
    return 0.5*(1+math.cos(math.pi*min(max(p,0.0),1.0)))

scheduler=torch.optim.lr_scheduler.LambdaLR(
    optimizer,lr_lambda=lr_mult
)

criterion=nn.CrossEntropyLoss(
    weight=class_w,
    label_smoothing=LABEL_SMOOTHING,
)

amp_enabled=bool(USE_AMP and DEVICE.type=="cuda")
scaler=torch.amp.GradScaler("cuda", enabled=amp_enabled)

print("Train batches/epoch:",len(train_loader))
print("Updates/epoch:",updates_per_epoch)
print("Total updates:",total_updates)
print("Warmup updates:",warmup_updates)
print("Effective batch:",MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS)
print("AMP:",amp_enabled)


AdamW fused: True
Train batches/epoch: 4212
Updates/epoch: 1053
Total updates: 3159
Warmup updates: 189
Effective batch: 16
AMP: True


In [13]:
# ============================================================
# CELL 12 — Metrics and evaluation
# ============================================================

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_true = []
    all_pred = []
    all_meta = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(input_ids, key_valid, lengths, pos)

        pred = logits.float().argmax(-1).cpu().tolist()
        all_pred.extend(pred)

        if "labels" in batch:
            all_true.extend(batch["labels"].tolist())

        all_meta.extend(zip(
            batch["doc_id"],
            batch["sent_idx"],
            batch["domain"]
        ))

    return all_true, all_pred, all_meta


def score_ids(y_true, y_pred, title="Combined"):
    label_ids = list(range(NUM_LABELS))

    macro_f1 = f1_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )
    macro_p = precision_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )
    macro_r = recall_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    print(f"{title}: F1={macro_f1:.6f} P={macro_p:.6f} R={macro_r:.6f}")
    return macro_f1, macro_p, macro_r



def dev_label_ceiling_diagnostic(y_true):
    present = sorted(set(y_true))
    absent = [i for i in range(NUM_LABELS) if i not in present]
    ceiling = len(present) / NUM_LABELS
    print(
        f"Dev contains {len(present)}/{NUM_LABELS} labels; "
        f"formal Macro-F1 ceiling from label absence = {ceiling:.6f}"
    )
    if absent:
        print("Absent dev labels:", [LABELS[i] for i in absent])
    return ceiling


def detailed_dev_eval(model, verbose=True):
    y_true, y_pred, meta = predict_loader(model, dev_loader)

    combined = score_ids(y_true, y_pred, "Combined CL+IT")
    dev_label_ceiling_diagnostic(y_true)

    # ---- diagnostics: collapse check + F1 over labels actually present in dev
    pred_c, gold_c = Counter(y_pred), Counter(y_true)
    present = sorted(set(y_true))
    f1_present = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    print(f"   Macro-F1 over {len(present)} labels present in dev: {f1_present:.6f}")
    print("   gold counts:", {LABELS[k]: gold_c[k] for k in present})
    print("   pred counts:", {LABELS[k]: pred_c[k] for k in sorted(pred_c)})
    if max(pred_c.values()) / max(1, len(y_pred)) > 0.60:
        print("   ⚠️ >60% of predictions are one class — model may be collapsing.")

    for dom in ["CL", "IT"]:
        idx = [i for i, x in enumerate(meta) if x[2] == dom]
        yt = [y_true[i] for i in idx]
        yp = [y_pred[i] for i in idx]
        score_ids(yt, yp, dom)

    if verbose:
        print("\nPer-label report (all 13 labels explicitly included):")
        print(classification_report(
            y_true, y_pred,
            labels=list(range(NUM_LABELS)),
            target_names=LABELS,
            digits=4,
            zero_division=0
        ))

    return combined, y_true, y_pred, meta

@torch.no_grad()
def predict_logits(model, loader):
    """Returns (logits[N,13] float32, meta list, y_true list) in loader order."""
    model.eval()
    outs, meta, y_true = [], [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            lg = model(input_ids, key_valid, lengths, pos)
        outs.append(lg.float().cpu().numpy())
        meta.extend(zip(batch["doc_id"], batch["sent_idx"], batch["domain"]))
        if "labels" in batch:
            y_true.extend(batch["labels"].tolist())
    return np.concatenate(outs, 0), meta, y_true


In [14]:
# ============================================================
# CELL 13 — Train rsLoRA; 4x/epoch DEV selection
# ============================================================
def fmt_sec(s):
    s=max(0,int(s)); h,s=divmod(s,3600); m,s=divmod(s,60)
    return f"{h:02d}:{m:02d}:{s:02d}"

best_f1=-1.0
best_epoch=None
best_frac=None
best_batch_in_epoch=None
best_global_update=None
bad_evals=0
global_update=0
history=[]
top_snapshots=[]  # list of dicts: f1, frac, path
train_start=time.time()

optimizer.zero_grad(set_to_none=True)

for epoch in range(1,NUM_EPOCHS+1):
    if hasattr(train_loader.batch_sampler,"set_epoch"):
        train_loader.batch_sampler.set_epoch(epoch)

    model.train()
    running=torch.zeros((),device=DEVICE,dtype=torch.float32)
    seen_batches=0
    ep0=time.time()

    # Align evals to optimizer-update boundaries so refit replay is exact.
    eval_steps=sorted(set(
        min(
            len(train_loader),
            max(GRAD_ACCUM_STEPS,
                int(round(len(train_loader)*j/EVAL_PER_EPOCH/GRAD_ACCUM_STEPS))*GRAD_ACCUM_STEPS)
        )
        for j in range(1,EVAL_PER_EPOCH+1)
    ))
    eval_steps[-1]=len(train_loader)
    if epoch==1:
        print("Dev evaluation at batches:",eval_steps,"of",len(train_loader))

    stop_training=False

    for step,batch in enumerate(train_loader,1):
        input_ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        key_valid=batch["key_valid"].to(DEVICE,non_blocking=True)
        lengths=batch["lengths"].to(DEVICE,non_blocking=True)
        pos=batch["pos"].to(DEVICE,non_blocking=True)
        labels=batch["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits=model(input_ids,key_valid,lengths,pos)
            loss=criterion(logits,labels)
            scaled_loss=loss/GRAD_ACCUM_STEPS

        if step%NAN_CHECK_EVERY==0 or step==len(train_loader):
            if not bool(torch.isfinite(loss.detach()).item()):
                raise FloatingPointError(
                    f"Non-finite loss epoch={epoch} batch={step}"
                )

        scaler.scale(scaled_loss).backward()
        running.add_(loss.detach().float())
        seen_batches+=1

        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                GRAD_CLIP,
            )
            old_scale=scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale=scaler.get_scale()
            optimizer.zero_grad(set_to_none=True)

            if new_scale>=old_scale:
                scheduler.step()
                global_update+=1

        if step%PRINT_EVERY==0 or step==len(train_loader):
            elapsed=time.time()-ep0
            rate=elapsed/max(step,1)
            eta_ep=rate*(len(train_loader)-step)
            completed=(epoch-1)+step/len(train_loader)
            eta_total=rate*len(train_loader)*max(0,NUM_EPOCHS-completed)
            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={float((running/max(seen_batches,1)).item()):.4f} "
                f"lr_rs={optimizer.param_groups[0]['lr']:.2e} "
                f"lr_head={optimizer.param_groups[1]['lr']:.2e} "
                f"elapsed={fmt_sec(elapsed)} "
                f"ETA_epoch={fmt_sec(eta_ep)} "
                f"ETA_maxtrain≈{fmt_sec(eta_total)}",
                flush=True,
            )

        if step in eval_steps:
            metrics,y_true,y_pred,meta=detailed_dev_eval(
                model,verbose=False
            )
            macro_f1,macro_p,macro_r=metrics
            frac=(epoch-1)+step/len(train_loader)

            pred_counts=Counter(y_pred)
            max_share=max(pred_counts.values())/max(1,len(y_pred))
            if max_share>0.65:
                print(
                    f"⚠️ Collapse warning: dominant predicted-class share={max_share:.3f}"
                )

            row={
                "epoch":epoch,
                "epoch_frac":float(frac),
                "batch_in_epoch":int(step),
                "global_update":int(global_update),
                "train_loss":float((running/max(seen_batches,1)).item()),
                "macro_f1":float(macro_f1),
                "macro_precision":float(macro_p),
                "macro_recall":float(macro_r),
                "elapsed_sec":time.time()-ep0,
            }
            history.append(row)
            print("Eval summary:",row)

            # Keep the strongest few heldout snapshots for a later within-rsLoRA ensemble.
            # We only serialize a state when it qualifies for the current top-k.
            qualifies_for_topk = (
                len(top_snapshots) < TOPK_HELDOUT_SNAPSHOTS
                or macro_f1 > min(x["f1"] for x in top_snapshots)
            )
            snapshot_state = None
            if qualifies_for_topk:
                snapshot_state = {
                    k:v.detach().cpu()
                    for k,v in model.state_dict().items()
                    if (
                        "rslora_A" in k or
                        "rslora_B" in k or
                        k.startswith(HEAD_PREFIXES)
                    )
                }
                snap_path = ROOT / (
                    f"heldout_snapshot_e{frac:.3f}_f1_{macro_f1:.6f}.pt"
                )
                torch.save({
                    "task_state":snapshot_state,
                    "epoch_frac":float(frac),
                    "macro_f1":float(macro_f1),
                    "batch_in_epoch":int(step),
                    "global_update":int(global_update),
                }, snap_path)

                top_snapshots.append({
                    "f1":float(macro_f1),
                    "frac":float(frac),
                    "path":str(snap_path),
                })
                top_snapshots.sort(key=lambda x:x["f1"], reverse=True)

                while len(top_snapshots) > TOPK_HELDOUT_SNAPSHOTS:
                    dropped = top_snapshots.pop(-1)
                    try:
                        os.remove(dropped["path"])
                    except FileNotFoundError:
                        pass

                print(
                    "Top snapshots:",
                    [(round(x["f1"],6), round(x["frac"],3)) for x in top_snapshots]
                )

            if macro_f1>best_f1:
                best_f1=float(macro_f1)
                best_epoch=int(math.ceil(frac))
                best_frac=float(frac)
                best_batch_in_epoch=int(step)
                best_global_update=int(global_update)
                bad_evals=0

                task_state = snapshot_state
                if task_state is None:
                    task_state={
                        k:v.detach().cpu()
                        for k,v in model.state_dict().items()
                        if (
                            "rslora_A" in k or
                            "rslora_B" in k or
                            k.startswith(HEAD_PREFIXES)
                        )
                    }

                torch.save({
                    "task_state":task_state,
                    "best_epoch":best_epoch,
                    "best_frac":best_frac,
                    "best_batch_in_epoch":best_batch_in_epoch,
                    "best_global_update":best_global_update,
                    "best_macro_f1":best_f1,
                    "history":history,
                    "labels":LABELS,
                    "label_to_id":LABEL_TO_ID,
                    "config":{
                        "experiment":"Model4 rsLoRA teacher search",
                        "r":RSLORA_R,
                        "alpha":RSLORA_ALPHA,
                        "scaling":RSLORA_SCALING,
                        "dropout":RSLORA_DROPOUT,
                        "targets":RSLORA_TARGETS,
                        "adapter_lr":RSLORA_LR,
                        "head_lr":HEAD_LR,
                        "max_len":MAX_LEN,
                        "context_sents":CONTEXT_SENTS,
                    },
                },BEST_CKPT)

                print(
                    f"✅ NEW BEST at epoch {frac:.3f} | "
                    f"batch={step} | update={global_update}: {BEST_CKPT}"
                )
            else:
                bad_evals+=1

            print(
                f"Best so far: epoch_frac={best_frac:.3f} "
                f"Macro-F1={best_f1:.6f} "
                f"| non-improving evals={bad_evals}\n"
            )

            if bad_evals>=EARLY_STOP_PATIENCE_EVALS:
                print(
                    f"⏹️ Early stopping after {bad_evals} "
                    "non-improving dev evaluations."
                )
                stop_training=True
                break

            model.train()

    if stop_training:
        break

with open(ROOT/"history.json","w",encoding="utf-8") as f:
    json.dump(history,f,indent=2)
with open(ROOT/"top_snapshots.json","w",encoding="utf-8") as f:
    json.dump(top_snapshots,f,indent=2)

assert best_frac is not None
assert top_snapshots
print("Training complete | elapsed:",fmt_sec(time.time()-train_start))
print(
    "BEST_EPOCH =",best_epoch,
    "| BEST_EPOCH_FRAC =",best_frac,
    "| BEST_BATCH =",best_batch_in_epoch,
    "| BEST_UPDATE =",best_global_update,
    "| BEST_MACRO_F1 =",best_f1
)


Dev evaluation at batches: [1052, 2104, 3160, 4212] of 4212
epoch=1 batch=250/4212 loss=2.3233 lr_rs=4.92e-05 lr_head=2.62e-04 elapsed=00:00:35 ETA_epoch=00:09:24 ETA_maxtrain≈00:29:23
epoch=1 batch=500/4212 loss=2.0662 lr_rs=9.92e-05 lr_head=5.29e-04 elapsed=00:01:09 ETA_epoch=00:08:36 ETA_maxtrain≈00:28:08
epoch=1 batch=750/4212 loss=1.9347 lr_rs=1.48e-04 lr_head=7.92e-04 elapsed=00:01:43 ETA_epoch=00:07:57 ETA_maxtrain≈00:27:18
epoch=1 batch=1000/4212 loss=1.8304 lr_rs=1.50e-04 lr_head=7.99e-04 elapsed=00:02:18 ETA_epoch=00:07:23 ETA_maxtrain≈00:26:46
Combined CL+IT: F1=0.288567 P=0.389620 R=0.279770
Dev contains 11/13 labels; formal Macro-F1 ceiling from label absence = 0.846154
Absent dev labels: ['Dissent', 'PrecedentOverruled']
   Macro-F1 over 11 labels present in dev: 0.341034
   gold counts: {'Fact': 444, 'Issue': 15, 'ArgumentPetitioner': 400, 'ArgumentRespondent': 272, 'Statute': 120, 'PrecedentReliedUpon': 254, 'PrecedentNotReliedUpon': 50, 'RulingByLowerCourt': 102, 'Rati

In [15]:
# ============================================================
# CELL 14 — Reload exact best rsLoRA checkpoint + DEV artifacts
# ============================================================
best_obj=torch.load(BEST_CKPT,map_location="cpu",weights_only=False)
missing,unexpected=model.load_state_dict(
    best_obj["task_state"],strict=False
)

bad_missing=[
    k for k in missing
    if (
        "rslora_A" in k or
        "rslora_B" in k or
        k.startswith(HEAD_PREFIXES)
    )
]
assert not bad_missing,bad_missing
assert not unexpected,unexpected

best_f1=float(best_obj["best_macro_f1"])
best_epoch=int(best_obj["best_epoch"])
best_frac=float(best_obj.get("best_frac",best_epoch))

print(
    "Reloaded best rsLoRA checkpoint:",
    "epoch_frac=",best_frac,
    "Macro-F1=",best_f1
)

dev_logits,dev_meta,dev_y=predict_logits(model,dev_loader)

np.savez_compressed(
    ROOT/"dev_logits.npz",
    logits=dev_logits,
    y_true=np.asarray(dev_y,dtype=np.int64),
    doc_id=np.asarray([x[0] for x in dev_meta],dtype=object),
    sent_idx=np.asarray([x[1] for x in dev_meta],dtype=np.int64),
    domain=np.asarray([x[2] for x in dev_meta],dtype=object),
)

def records_to_prediction(meta,pred_ids):
    by_doc={}
    for (did,si,dom),pid in zip(meta,pred_ids):
        by_doc.setdefault(did,[]).append((si,LABELS[int(pid)]))
    out={}
    for did,items in by_doc.items():
        items.sort(key=lambda x:x[0])
        out[did]=[lab for _,lab in items]
    return out

dev_pred=records_to_prediction(
    dev_meta,dev_logits.argmax(-1)
)
with open(DEV_PRED,"w",encoding="utf-8") as f:
    json.dump(dev_pred,f,ensure_ascii=False,indent=2)

def write_truth(docs,path):
    truth={}
    for d in docs:
        did=normalize_id(d["id"])
        truth[did]=list(d["labels"])
    with open(path,"w",encoding="utf-8") as f:
        json.dump(truth,f,ensure_ascii=False,indent=2)

write_truth(cl_dev_raw,CL_DEV_TRUTH)
write_truth(it_dev_raw,IT_DEV_TRUTH)

with open(LABELS_JSON,"w",encoding="utf-8") as f:
    json.dump(LABELS,f,ensure_ascii=False,indent=2)

for p in [CL_DEV_TRUTH,IT_DEV_TRUTH,LABELS_JSON,DEV_PRED]:
    assert Path(p).is_file(),p

print("✅ DEV prediction/truth/labels written.")


Reloaded best rsLoRA checkpoint: epoch_frac= 2.0 Macro-F1= 0.4027961783131959
✅ DEV prediction/truth/labels written.


In [16]:
# ============================================================
# CELL 15 — EXACT local Phase-3 evaluator script supplied by course
# ============================================================

EVALUATOR_CODE = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument("--truth-cl", required=True)
    parser.add_argument("--truth-it", required=True)
    parser.add_argument("--prediction", required=True)
    parser.add_argument("--labels", required=True)
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [f"missing={len(missing)}", f"extra={len(extra)}"]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
'''

with open(EVAL_SCRIPT, "w", encoding="utf-8") as f:
    f.write(EVALUATOR_CODE)

print("Saved exact evaluator to:", EVAL_SCRIPT)

Saved exact evaluator to: /kaggle/working/phase3_model4_rslora_teacher/evaluate_phase3.py


In [17]:
# ============================================================
# CELL 16 — RUN THE OFFICIAL-STYLE LOCAL EVALUATOR
# Do not continue to final refit until this passes.
# ============================================================

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--truth-cl", str(CL_DEV_TRUTH),
    "--truth-it", str(IT_DEV_TRUTH),
    "--prediction", str(DEV_PRED),
    "--labels", str(LABELS_JSON),
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\nSTDOUT:\n", result.stdout)
if result.stderr:
    print("\nSTDERR:\n", result.stderr)

assert result.returncode == 0, "❌ Phase-3 evaluator failed."
print("✅ DEV prediction passed the exact Phase-3 evaluator.")

Running:
/usr/bin/python3 /kaggle/working/phase3_model4_rslora_teacher/evaluate_phase3.py --truth-cl /kaggle/working/phase3_model4_rslora_teacher/cl_dev_truth.json --truth-it /kaggle/working/phase3_model4_rslora_teacher/it_dev_truth.json --prediction /kaggle/working/phase3_model4_rslora_teacher/prediction_dev.json --labels /kaggle/working/phase3_model4_rslora_teacher/labels.json

STDOUT:
 CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.402796
Macro-Precision: 0.421110
Macro-Recall:    0.407628

✅ DEV prediction passed the exact Phase-3 evaluator.


In [18]:
# ============================================================
# CELL 16b — Optional transition-aware decoding tuned on DEV
#
# Improvements:
#   * CL and IT transition/prior statistics estimated separately
#   * additive smoothing
#   * no hard "impossible label" bans
#   * only adopt decoding when DEV Macro-F1 improves by >= 0.005
# ============================================================

from sklearn.metrics import f1_score as _f1

def transition_matrix(docs_list, smooth=1.0):
    T = np.full((NUM_LABELS, NUM_LABELS), float(smooth), dtype=np.float64)
    for d in docs_list:
        ids = [LABEL_TO_ID[x] for x in d["labels"]]
        for a, b in zip(ids[:-1], ids[1:]):
            T[a, b] += 1.0
    T /= T.sum(1, keepdims=True)
    return np.log(T)

def class_log_prior(docs_list, smooth=1.0):
    c = np.full(NUM_LABELS, float(smooth), dtype=np.float64)
    for d in docs_list:
        for x in d["labels"]:
            c[LABEL_TO_ID[x]] += 1.0
    return np.log(c / c.sum())

def log_softmax_np(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

def viterbi_decode(logp, trans, beta):
    n, K = logp.shape
    score = logp[0].copy()
    back = np.zeros((n, K), dtype=np.int32)
    tr = beta * trans

    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand[back[t], np.arange(K)] + logp[t]

    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())

    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]

    return path

def build_domain_stats(cl_docs, it_docs):
    return {
        "CL": {
            "trans": transition_matrix(cl_docs, smooth=1.0),
            "prior": class_log_prior(cl_docs, smooth=1.0),
        },
        "IT": {
            "trans": transition_matrix(it_docs, smooth=1.0),
            "prior": class_log_prior(it_docs, smooth=1.0),
        },
    }

def decode_docs(logits, meta, domain_stats, alpha, beta):
    # logits aligned with meta=(doc_id, sent_idx, domain)
    by_doc = {}
    for i, (did, si, dom) in enumerate(meta):
        by_doc.setdefault(did, []).append((si, i, dom))

    pred = np.zeros(len(meta), dtype=np.int64)

    for did, items in by_doc.items():
        items.sort(key=lambda x: x[0])
        idx = [i for _, i, _ in items]
        dom = items[0][2]
        stats = domain_stats[dom]

        lp = (
            log_softmax_np(logits[idx].astype(np.float64))
            - alpha * stats["prior"][None, :]
        )

        pred[idx] = (
            viterbi_decode(lp, stats["trans"], beta)
            if beta > 0
            else lp.argmax(-1)
        )

    return pred

# Tune only on held-out DEV predictions.
dev_logits, dev_meta, dev_y = predict_logits(model, dev_loader)

domain_stats_train = build_domain_stats(
    cl_train_raw,
    it_train_raw,
)

def dev_f1(alpha, beta):
    p = decode_docs(
        dev_logits,
        dev_meta,
        domain_stats_train,
        alpha,
        beta
    )
    return _f1(
        dev_y,
        p,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0
    )

rows = []
for alpha in [-0.25, 0.0, 0.25, 0.5, 0.75]:
    for beta in [0.0, 0.25, 0.5, 1.0]:
        rows.append((dev_f1(alpha, beta), alpha, beta))

base_f1 = dev_f1(0.0, 0.0)
rows.sort(key=lambda r: (-r[0], r[2], abs(r[1])))

print(f"Baseline argmax (alpha=0, beta=0): {base_f1:.6f}")
print("Top decoding settings (dev F1, alpha, beta):")
for r in rows[:8]:
    print(f"  {r[0]:.6f}  alpha={r[1]:+.2f}  beta={r[2]:.2f}")

if rows[0][0] - base_f1 >= 0.005:
    DECODE_ALPHA, DECODE_BETA = rows[0][1], rows[0][2]
else:
    DECODE_ALPHA, DECODE_BETA = 0.0, 0.0

print("SELECTED decoding: alpha=", DECODE_ALPHA, "beta=", DECODE_BETA)

dec_ids = decode_docs(
    dev_logits,
    dev_meta,
    domain_stats_train,
    DECODE_ALPHA,
    DECODE_BETA
)

dev_pred_dec = records_to_prediction(dev_meta, dec_ids)
DEV_PRED_DEC = ROOT / "prediction_dev_decoded.json"

with open(DEV_PRED_DEC, "w", encoding="utf-8") as f:
    json.dump(dev_pred_dec, f, ensure_ascii=False, indent=2)

# Run the exact supplied evaluator again on the decoded DEV prediction.
res = subprocess.run(
    [
        sys.executable,
        str(EVAL_SCRIPT),
        "--truth-cl", str(CL_DEV_TRUTH),
        "--truth-it", str(IT_DEV_TRUTH),
        "--prediction", str(DEV_PRED_DEC),
        "--labels", str(LABELS_JSON),
    ],
    capture_output=True,
    text=True
)

print(res.stdout)
if res.stderr:
    print(res.stderr)

assert res.returncode == 0, "Decoded DEV prediction failed the exact evaluator."
print("✅ Decoded DEV prediction passed the exact Phase-3 evaluator.")

Baseline argmax (alpha=0, beta=0): 0.402796
Top decoding settings (dev F1, alpha, beta):
  0.420592  alpha=+0.00  beta=1.00
  0.419735  alpha=+0.25  beta=1.00
  0.412929  alpha=+0.25  beta=0.50
  0.412706  alpha=+0.50  beta=1.00
  0.411799  alpha=-0.25  beta=1.00
  0.409761  alpha=+0.25  beta=0.25
  0.409562  alpha=+0.00  beta=0.25
  0.405367  alpha=-0.25  beta=0.25
SELECTED decoding: alpha= 0.0 beta= 1.0
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.420592
Macro-Precision: 0.528295
Macro-Recall:    0.398690

✅ Decoded DEV prediction passed the exact Phase-3 evaluator.


In [19]:
# ============================================================
# CELL 18 — Fresh Phase-1 Model4 + rsLoRA for fractional train+dev refit
# ============================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BEST_EPOCH=int(best_obj["best_epoch"])
BEST_FRAC=float(best_obj.get("best_frac",BEST_EPOCH))
BEST_BATCH=int(best_obj.get("best_batch_in_epoch",0))
BEST_UPDATE=int(best_obj.get("best_global_update",0))

assert BEST_EPOCH>=1 and 0.0<BEST_FRAC<=NUM_EPOCHS
print(
    "Final refit target:",
    f"epoch_frac={BEST_FRAC:.3f}",
    f"| selected_batch={BEST_BATCH}",
    f"| selected_update={BEST_UPDATE}",
)

all_labeled_records=(
    make_labeled_records(cl_train_raw,"CL")
    +make_labeled_records(it_train_raw,"IT")
    +make_labeled_records(cl_dev_raw,"CL")
    +make_labeled_records(it_dev_raw,"IT")
)

all_counts=Counter(x["label"] for x in all_labeled_records)
N_all=len(all_labeled_records)
raw_w_all=np.array([
    math.sqrt(N_all/(NUM_LABELS*max(all_counts[lab],1)))
    for lab in LABELS
],dtype=np.float32)
raw_w_all/=raw_w_all.mean()

class_w_all=torch.tensor(
    np.clip(raw_w_all,CLASS_WEIGHT_MIN,CLASS_WEIGHT_MAX),
    dtype=torch.float32,
    device=DEVICE,
)

def build_fresh_rr_from_phase1():
    seed_everything(SEED)
    bb=build_backbone()

    o=torch.load(
        PHASE1_CKPT,
        map_location="cpu",
        weights_only=False,
    )
    st=o["model"] if isinstance(o,dict) and "model" in o else o
    bb.load_state_dict(st,strict=True)

    inject_rslora(
        bb,
        targets=RSLORA_TARGETS,
        r=RSLORA_R,
        alpha=RSLORA_ALPHA,
        dropout=RSLORA_DROPOUT,
    )

    m=RRClassifier(bb,NUM_LABELS,HEAD_DROPOUT)

    for n,p in m.named_parameters():
        if n.startswith(HEAD_PREFIXES):
            p.requires_grad=True

    for n,p in m.named_parameters():
        if p.requires_grad:
            assert (
                "rslora_A" in n or
                "rslora_B" in n or
                n.startswith(HEAD_PREFIXES)
            ),n

    return m.to(DEVICE)

final_model=build_fresh_rr_from_phase1()

all_ds=RRDataset(all_labeled_records,labeled=True)
all_lengths=[
    len(build_context_ids(r["domain"],r["prev"],r["target"],r["next"]))
    for r in all_labeled_records
]

all_loader=DataLoader(
    all_ds,
    batch_sampler=SortishBatchSampler(
        all_lengths,MICRO_BATCH_SIZE,seed=SEED
    ),
    collate_fn=collate_rr,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type=="cuda"),
    persistent_workers=NUM_WORKERS>0,
)

final_adapter=[]
final_head=[]
for n,p in final_model.named_parameters():
    if not p.requires_grad:
        continue
    if "rslora_A" in n or "rslora_B" in n:
        final_adapter.append(p)
    else:
        assert n.startswith(HEAD_PREFIXES),n
        final_head.append(p)

final_groups=[
    {"params":final_adapter,"lr":RSLORA_LR,"weight_decay":0.0},
    {"params":final_head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
]
try:
    final_optimizer=torch.optim.AdamW(
        final_groups,fused=(DEVICE.type=="cuda")
    )
except Exception:
    final_optimizer=torch.optim.AdamW(final_groups)

REFIT_TOTAL_BATCHES=int(
    round(BEST_FRAC*len(all_loader)/GRAD_ACCUM_STEPS)
)*GRAD_ACCUM_STEPS
REFIT_TOTAL_BATCHES=max(
    GRAD_ACCUM_STEPS,REFIT_TOTAL_BATCHES
)

final_updates_per_epoch=math.ceil(
    len(all_loader)/GRAD_ACCUM_STEPS
)
SCHED_EPOCHS=(
    NUM_EPOCHS if REFIT_MATCH_ORIGINAL_SCHEDULE
    else max(1,math.ceil(BEST_FRAC))
)
final_total_updates=SCHED_EPOCHS*final_updates_per_epoch
final_warmup=max(
    1,int(final_total_updates*WARMUP_RATIO)
)

def final_lr_mult(step):
    if step<final_warmup:
        return max(1e-8,step/final_warmup)
    p=(step-final_warmup)/max(
        1,final_total_updates-final_warmup
    )
    return 0.5*(
        1+math.cos(math.pi*min(max(p,0.0),1.0))
    )

final_scheduler=torch.optim.lr_scheduler.LambdaLR(
    final_optimizer,
    lr_lambda=final_lr_mult,
)

final_criterion=nn.CrossEntropyLoss(
    weight=class_w_all,
    label_smoothing=LABEL_SMOOTHING,
)

final_scaler=torch.amp.GradScaler(
    "cuda",enabled=amp_enabled
)

print(
    "All labeled records:",len(all_labeled_records),
    "| refit total micro-batches:",REFIT_TOTAL_BATCHES,
    "| loader batches/epoch:",len(all_loader),
)


Final refit target: epoch_frac=2.000 | selected_batch=4212 | selected_update=2101
All labeled records: 18987 | refit total micro-batches: 9496 | loader batches/epoch: 4747


In [20]:
# ============================================================
# CELL 19 — Final rsLoRA fractional refit + ETA
# ============================================================
final_optimizer.zero_grad(set_to_none=True)

refit_start=time.time()
refit_updates=0
seen_batches=0
done=False

max_refit_epochs=int(
    math.ceil(REFIT_TOTAL_BATCHES/len(all_loader))
)+1

for epoch in range(1,max_refit_epochs+1):
    if hasattr(all_loader.batch_sampler,"set_epoch"):
        all_loader.batch_sampler.set_epoch(epoch)

    final_model.train()
    running=torch.zeros(
        (),device=DEVICE,dtype=torch.float32
    )
    n_batches=0

    for step,batch in enumerate(all_loader,1):
        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True
            break

        input_ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        key_valid=batch["key_valid"].to(DEVICE,non_blocking=True)
        lengths=batch["lengths"].to(DEVICE,non_blocking=True)
        pos=batch["pos"].to(DEVICE,non_blocking=True)
        labels=batch["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits=final_model(
                input_ids,key_valid,lengths,pos
            )
            loss=final_criterion(logits,labels)
            scaled=loss/GRAD_ACCUM_STEPS

        if (
            seen_batches%NAN_CHECK_EVERY==0
            and not bool(torch.isfinite(loss.detach()).item())
        ):
            raise FloatingPointError("Non-finite refit loss")

        final_scaler.scale(scaled).backward()
        running.add_(loss.detach().float())
        n_batches+=1
        seen_batches+=1

        do_update=(
            seen_batches%GRAD_ACCUM_STEPS==0
            or seen_batches>=REFIT_TOTAL_BATCHES
        )

        if do_update:
            final_scaler.unscale_(final_optimizer)
            torch.nn.utils.clip_grad_norm_(
                [
                    p for p in final_model.parameters()
                    if p.requires_grad
                ],
                GRAD_CLIP,
            )
            old_scale=final_scaler.get_scale()
            final_scaler.step(final_optimizer)
            final_scaler.update()
            new_scale=final_scaler.get_scale()
            final_optimizer.zero_grad(set_to_none=True)

            if new_scale>=old_scale:
                final_scheduler.step()
                refit_updates+=1

        if (
            seen_batches%PRINT_EVERY==0
            or seen_batches>=REFIT_TOTAL_BATCHES
        ):
            elapsed=time.time()-refit_start
            rate=elapsed/max(seen_batches,1)
            eta=rate*(REFIT_TOTAL_BATCHES-seen_batches)
            print(
                f"refit batch={seen_batches}/{REFIT_TOTAL_BATCHES} "
                f"epoch≈{seen_batches/len(all_loader):.3f} "
                f"loss={float((running/max(n_batches,1)).item()):.4f} "
                f"elapsed={fmt_sec(elapsed)} "
                f"ETA≈{fmt_sec(eta)}",
                flush=True,
            )

        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True
            break

    if done:
        break

assert seen_batches==REFIT_TOTAL_BATCHES,(
    seen_batches,REFIT_TOTAL_BATCHES
)

final_task_state={
    k:v.detach().cpu()
    for k,v in final_model.state_dict().items()
    if (
        "rslora_A" in k or
        "rslora_B" in k or
        k.startswith(HEAD_PREFIXES)
    )
}

torch.save({
    "task_state":final_task_state,
    "source_best_epoch_frac":BEST_FRAC,
    "source_best_batch_in_epoch":BEST_BATCH,
    "source_best_global_update":BEST_UPDATE,
    "refit_total_batches":REFIT_TOTAL_BATCHES,
    "refit_optimizer_updates":refit_updates,
    "source_best_dev_f1":best_f1,
    "labels":LABELS,
    "label_to_id":LABEL_TO_ID,
    "config":{
        "experiment":"Model4 rsLoRA final refit",
        "r":RSLORA_R,
        "alpha":RSLORA_ALPHA,
        "scaling":RSLORA_SCALING,
        "targets":RSLORA_TARGETS,
    },
},FINAL_CKPT)

print(
    "✅ Final refit saved:",FINAL_CKPT,
    "| elapsed:",fmt_sec(time.time()-refit_start)
)


refit batch=250/9496 epoch≈0.053 loss=2.3338 elapsed=00:00:36 ETA≈00:22:29
refit batch=500/9496 epoch≈0.105 loss=2.1213 elapsed=00:01:11 ETA≈00:21:21
refit batch=750/9496 epoch≈0.158 loss=1.9527 elapsed=00:01:45 ETA≈00:20:35
refit batch=1000/9496 epoch≈0.211 loss=1.8651 elapsed=00:02:21 ETA≈00:20:01
refit batch=1250/9496 epoch≈0.263 loss=1.7926 elapsed=00:02:57 ETA≈00:19:28
refit batch=1500/9496 epoch≈0.316 loss=1.7413 elapsed=00:03:34 ETA≈00:19:02
refit batch=1750/9496 epoch≈0.369 loss=1.6970 elapsed=00:04:12 ETA≈00:18:39
refit batch=2000/9496 epoch≈0.421 loss=1.6580 elapsed=00:04:48 ETA≈00:18:00
refit batch=2250/9496 epoch≈0.474 loss=1.6214 elapsed=00:05:24 ETA≈00:17:24
refit batch=2500/9496 epoch≈0.527 loss=1.5833 elapsed=00:06:01 ETA≈00:16:50
refit batch=2750/9496 epoch≈0.579 loss=1.5461 elapsed=00:06:38 ETA≈00:16:17
refit batch=3000/9496 epoch≈0.632 loss=1.5193 elapsed=00:07:12 ETA≈00:15:37
refit batch=3250/9496 epoch≈0.685 loss=1.4934 elapsed=00:07:47 ETA≈00:14:57
refit batch=350

In [21]:
# ============================================================
# CELL 20 — Build test sentence records
# ============================================================

def make_test_records(test_dict, domain):
    records = []
    for raw_id, sents in test_dict.items():
        did = normalize_id(raw_id)
        assert isinstance(sents, list)
        for i, sent in enumerate(sents):
            records.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
            })
    return records

test_records = (
    make_test_records(cl_test_raw, "CL")
    + make_test_records(it_test_raw, "IT")
)

test_loader = DataLoader(
    RRDataset(test_records, labeled=False),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
    num_workers=2,
    pin_memory=True,
)

print("Total test sentence records:", len(test_records))
assert len(test_records) == (
    sum(len(v) for v in cl_test_raw.values())
    + sum(len(v) for v in it_test_raw.values())
)

Total test sentence records: 2197


In [22]:
# ============================================================
# CELL 21 — Test inference: heldout / refit / ensemble
# ============================================================
@torch.no_grad()
def predict_test_logits(m,loader):
    m.eval()
    outs,meta=[],[]

    for batch in loader:
        input_ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        key_valid=batch["key_valid"].to(DEVICE,non_blocking=True)
        lengths=batch["lengths"].to(DEVICE,non_blocking=True)
        pos=batch["pos"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            lg=m(input_ids,key_valid,lengths,pos)

        outs.append(lg.float().cpu().numpy())
        meta.extend(zip(
            batch["doc_id"],
            batch["sent_idx"],
            batch["domain"],
        ))

    return np.concatenate(outs,0),meta


domain_stats_all=build_domain_stats(
    cl_train_raw+cl_dev_raw,
    it_train_raw+it_dev_raw,
)

held_logits,test_meta=predict_test_logits(
    model,test_loader
)
refit_logits,test_meta_refit=predict_test_logits(
    final_model,test_loader
)
assert test_meta==test_meta_refit

np.savez_compressed(
    ROOT/"test_logits.npz",
    heldout=held_logits,
    refit=refit_logits,
    doc_id=np.asarray([x[0] for x in test_meta],dtype=object),
    sent_idx=np.asarray([x[1] for x in test_meta],dtype=np.int64),
    domain=np.asarray([x[2] for x in test_meta],dtype=object),
)

def pred_from_logits(lg,name):
    ids=decode_docs(
        lg,test_meta,domain_stats_all,
        DECODE_ALPHA,DECODE_BETA
    )
    pred=records_to_prediction(test_meta,ids)
    cnt=Counter(ids.tolist())
    print(
        f"[{name}] counts:",
        {LABELS[k]:cnt[k] for k in sorted(cnt)}
    )
    return pred

predictions={
    "heldout":pred_from_logits(
        held_logits,"heldout"
    ),
    "refit":pred_from_logits(
        refit_logits,"refit"
    ),
}

if ENSEMBLE_HELDOUT_AND_REFIT:
    ensemble_logits=0.5*held_logits+0.5*refit_logits
    predictions["ensemble"]=pred_from_logits(
        ensemble_logits,"ensemble"
    )

# Optional within-technique snapshot ensembles.
snapshot_mean_logits = None
if MAKE_SNAPSHOT_ENSEMBLES:
    top_meta_path = ROOT/"top_snapshots.json"
    if top_meta_path.is_file():
        with open(top_meta_path,"r",encoding="utf-8") as f:
            top_meta=json.load(f)

        snapshot_logits=[]
        for info in top_meta[:TOPK_HELDOUT_SNAPSHOTS]:
            sp=Path(info["path"])
            if not sp.is_file():
                print("skip missing snapshot:",sp)
                continue

            snap_model=build_fresh_rr_from_phase1()
            sobj=torch.load(sp,map_location="cpu",weights_only=False)
            miss,unexp=snap_model.load_state_dict(
                sobj["task_state"],strict=False
            )
            bad=[
                k for k in miss
                if (
                    "rslora_A" in k or
                    "rslora_B" in k or
                    k.startswith(HEAD_PREFIXES)
                )
            ]
            assert not bad and not unexp,(bad,unexp)

            lg,meta_snap=predict_test_logits(
                snap_model,test_loader
            )
            assert meta_snap==test_meta
            snapshot_logits.append(lg)

            del snap_model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if len(snapshot_logits)>=2:
            snapshot_mean_logits=np.mean(
                np.stack(snapshot_logits,axis=0),
                axis=0,
            )
            predictions["snapshot_ensemble"]=pred_from_logits(
                snapshot_mean_logits,
                "snapshot_ensemble",
            )
            predictions["snapshot_refit_ensemble"]=pred_from_logits(
                0.5*snapshot_mean_logits+0.5*refit_logits,
                "snapshot_refit_ensemble",
            )
            np.savez_compressed(
                ROOT/"snapshot_test_logits.npz",
                mean=snapshot_mean_logits.astype(np.float32),
                n_snapshots=np.asarray([len(snapshot_logits)],dtype=np.int64),
            )
        else:
            print("Snapshot ensemble skipped: fewer than 2 usable snapshots.")

prediction=predictions["refit"]

with open(OFFICIAL_PRED,"w",encoding="utf-8") as f:
    json.dump(
        prediction,f,
        ensure_ascii=False,indent=2
    )

print("Saved:",OFFICIAL_PRED)
print("Prediction documents:",len(prediction))


[heldout] counts: {'Fact': 456, 'Issue': 5, 'ArgumentPetitioner': 165, 'ArgumentRespondent': 133, 'Statute': 171, 'Dissent': 12, 'PrecedentReliedUpon': 336, 'PrecedentNotReliedUpon': 18, 'RulingByLowerCourt': 121, 'RatioOfTheDecision': 725, 'RulingByPresentCourt': 36, 'None': 19}
[refit] counts: {'Fact': 440, 'Issue': 4, 'ArgumentPetitioner': 325, 'ArgumentRespondent': 13, 'Statute': 205, 'Dissent': 9, 'PrecedentReliedUpon': 338, 'PrecedentNotReliedUpon': 8, 'RulingByLowerCourt': 163, 'RatioOfTheDecision': 666, 'RulingByPresentCourt': 18, 'None': 8}
[ensemble] counts: {'Fact': 450, 'Issue': 4, 'ArgumentPetitioner': 280, 'ArgumentRespondent': 26, 'Statute': 187, 'Dissent': 11, 'PrecedentReliedUpon': 340, 'PrecedentNotReliedUpon': 8, 'RulingByLowerCourt': 128, 'RatioOfTheDecision': 721, 'RulingByPresentCourt': 27, 'None': 15}
[snapshot_ensemble] counts: {'Fact': 456, 'Issue': 4, 'ArgumentPetitioner': 251, 'ArgumentRespondent': 58, 'Statute': 175, 'Dissent': 8, 'PrecedentReliedUpon': 338,

In [23]:
# ============================================================
# CELL 22 — Strict structural checks + submission ZIP variants
# ============================================================
def normalized_test_map(raw):
    out={}
    for raw_id,sents in raw.items():
        did=normalize_id(raw_id)
        assert did not in out,f"Duplicate normalized ID: {did}"
        out[did]=sents
    return out

cl_test=normalized_test_map(cl_test_raw)
it_test=normalized_test_map(it_test_raw)
assert not(set(cl_test)&set(it_test))

expected={}
expected.update(cl_test)
expected.update(it_test)
allowed=set(LABELS)

def validate_test_prediction(pred):
    assert isinstance(pred,dict)
    assert set(pred)==set(expected),(
        f"missing={set(expected)-set(pred)}, "
        f"extra={set(pred)-set(expected)}"
    )
    for did,sents in expected.items():
        assert isinstance(pred[did],list),did
        assert len(pred[did])==len(sents),(
            did,len(pred[did]),len(sents)
        )
        assert all(x in allowed for x in pred[did]),did
    return True

for name,pred in predictions.items():
    assert validate_test_prediction(pred)

    jp=ROOT/f"prediction_{name}.json"
    zp=ROOT/f"submission_{name}.zip"

    with open(jp,"w",encoding="utf-8") as f:
        json.dump(pred,f,ensure_ascii=False,indent=2)

    with zipfile.ZipFile(
        zp,"w",zipfile.ZIP_DEFLATED
    ) as z:
        z.write(jp,arcname="prediction.json")

    with zipfile.ZipFile(zp,"r") as z:
        assert z.namelist()==["prediction.json"]
        chk=json.loads(
            z.read("prediction.json").decode("utf-8")
        )

    assert chk==pred
    print("✅",zp.name)

# Standard submission.zip == refit for consistency.
with zipfile.ZipFile(
    SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED
) as z:
    z.write(
        ROOT/"prediction_refit.json",
        arcname="prediction.json",
    )

with zipfile.ZipFile(SUBMISSION_ZIP,"r") as z:
    assert z.namelist()==["prediction.json"]

assert len(prediction)==10
assert sum(len(v) for v in prediction.values())==2197

manifest={
    "experiment":"Phase3_Model4_rsLoRA_TeacherSearch",
    "phase1_model":"Model4_Efficient_GQA_RoPE_RMSNorm_SwiGLU",
    "technique":"rsLoRA",
    "targets":RSLORA_TARGETS,
    "r":RSLORA_R,
    "alpha":RSLORA_ALPHA,
    "scaling_rule":"alpha/sqrt(r)",
    "scaling":RSLORA_SCALING,
    "dropout":RSLORA_DROPOUT,
    "adapter_lr":RSLORA_LR,
    "head_lr":HEAD_LR,
    "label_smoothing":LABEL_SMOOTHING,
    "best_epoch":int(best_obj["best_epoch"]),
    "best_epoch_frac":float(
        best_obj.get("best_frac",best_obj["best_epoch"])
    ),
    "best_batch_in_epoch":int(
        best_obj.get("best_batch_in_epoch",0)
    ),
    "best_global_update":int(
        best_obj.get("best_global_update",0)
    ),
    "best_dev_macro_f1":float(
        best_obj["best_macro_f1"]
    ),
    "decode_alpha":float(DECODE_ALPHA),
    "decode_beta":float(DECODE_BETA),
    "effective_batch_size":(
        MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS
    ),
    "topk_heldout_snapshots":TOPK_HELDOUT_SNAPSHOTS,
    "snapshot_ensembles_enabled":MAKE_SNAPSHOT_ENSEMBLES,
    "seed":SEED,
    "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
    "phase1_checkpoint":str(PHASE1_CKPT),
    "artifacts":sorted([
        p.name for p in ROOT.iterdir()
        if p.is_file()
    ]),
}

with open(
    ROOT/"experiment_manifest.json",
    "w",encoding="utf-8"
) as f:
    json.dump(manifest,f,indent=2)

print("✅ FINAL STRUCTURAL CONTRACT PASSED")
print("Documents:",len(prediction))
print("Sentences:",sum(len(v) for v in prediction.values()))
print("submission.zip == refit")
print(
    "Variants:",
    ", ".join(f"submission_{k}.zip" for k in predictions)
)
print("Saved manifest:",ROOT/"experiment_manifest.json")


✅ submission_heldout.zip
✅ submission_refit.zip
✅ submission_ensemble.zip
✅ submission_snapshot_ensemble.zip
✅ submission_snapshot_refit_ensemble.zip
✅ FINAL STRUCTURAL CONTRACT PASSED
Documents: 10
Sentences: 2197
submission.zip == refit
Variants: submission_heldout.zip, submission_refit.zip, submission_ensemble.zip, submission_snapshot_ensemble.zip, submission_snapshot_refit_ensemble.zip
Saved manifest: /kaggle/working/phase3_model4_rslora_teacher/experiment_manifest.json


In [24]:
# ============================================================
# CELL 23 — Export rsLoRA teacher logits for later KD
# ============================================================
# This cell is OPTIONAL for rsLoRA scoring, but valuable if rsLoRA becomes
# the strongest teacher. It does not affect submission ZIPs.

def export_teacher_logits(m,records,filename):
    ds=RRDataset(records,labeled=True)
    loader=DataLoader(
        ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
    )

    logits,meta,y=predict_logits(m,loader)
    assert len(logits)==len(records)==len(meta)

    path=ROOT/filename
    np.savez_compressed(
        path,
        logits=logits.astype(np.float32),
        y=np.asarray(y,dtype=np.int64),
        doc_id=np.asarray(
            [x[0] for x in meta],dtype=object
        ),
        sent_idx=np.asarray(
            [x[1] for x in meta],dtype=np.int64
        ),
        domain=np.asarray(
            [x[2] for x in meta],dtype=object
        ),
    )

    acc=float(
        (
            logits.argmax(-1)
            ==np.asarray(y,dtype=np.int64)
        ).mean()
    )

    print(
        "✅ Saved:",path,
        "| rows:",len(logits),
        "| argmax accuracy:",round(acc,4)
    )
    return path

teacher_train_path=export_teacher_logits(
    model,
    train_records,
    "teacher_logits_train_heldout.npz",
)

teacher_refit_path=export_teacher_logits(
    final_model,
    all_labeled_records,
    "teacher_logits_train_dev_refit.npz",
)

print("KD teacher files ready:")
print(" ",teacher_train_path)
print(" ",teacher_refit_path)


✅ Saved: /kaggle/working/phase3_model4_rslora_teacher/teacher_logits_train_heldout.npz | rows: 16845 | argmax accuracy: 0.9628
✅ Saved: /kaggle/working/phase3_model4_rslora_teacher/teacher_logits_train_dev_refit.npz | rows: 18987 | argmax accuracy: 0.9628
KD teacher files ready:
  /kaggle/working/phase3_model4_rslora_teacher/teacher_logits_train_heldout.npz
  /kaggle/working/phase3_model4_rslora_teacher/teacher_logits_train_dev_refit.npz


In [25]:

# ============================================================
# CELL 24 — OPTIONAL cross-run TEST ensemble (safe-by-default)
# ============================================================
# IMPORTANT:
# - This cell can improve leaderboard score by combining Model-4 runs.
# - It does NOT create KD training targets. KD needs TRAIN-set teacher logits.
# - Only explicitly listed files are used. We intentionally do NOT auto-glob all
#   Kaggle test_logits.npz files, because that could silently mix unrelated models.
#
# Add paths only for Model-4 runs you intentionally want to combine, e.g.
# AdaLoRA Exp5 or LoRA Exp1.

EXTRA_MODEL4_TEST_LOGITS = [
    # "/kaggle/input/your-adalora-artifacts/test_logits.npz",
    # "/kaggle/input/your-lora-artifacts/test_logits.npz",
]

def softmax_np(x):
    x=np.asarray(x,dtype=np.float64)
    x=x-x.max(-1,keepdims=True)
    e=np.exp(x)
    return e/e.sum(-1,keepdims=True)

def align_external_rows(z,target_meta):
    assert "doc_id" in z and "sent_idx" in z and "domain" in z
    src_keys=[
        (normalize_id(str(d)),int(s),str(dom))
        for d,s,dom in zip(z["doc_id"],z["sent_idx"],z["domain"])
    ]
    tgt_keys=[
        (normalize_id(str(d)),int(s),str(dom))
        for d,s,dom in target_meta
    ]
    src_map={k:i for i,k in enumerate(src_keys)}
    assert len(src_map)==len(src_keys),"duplicate external keys"
    order=[src_map.get(k) for k in tgt_keys]
    assert all(i is not None for i in order),"external test ids do not match"
    return np.asarray(order,dtype=np.int64)

run_probs={}
run_probs["rsLoRA(this run)"]=0.5*softmax_np(held_logits)+0.5*softmax_np(refit_logits)

for path in EXTRA_MODEL4_TEST_LOGITS:
    try:
        z=np.load(path,allow_pickle=True)
        order=align_external_rows(z,test_meta)

        arrays=[]
        for key in ("heldout","refit"):
            if key in z:
                a=np.asarray(z[key])
                if a.shape==(len(test_meta),NUM_LABELS):
                    arrays.append(softmax_np(a[order]))

        if not arrays:
            print("skip (no usable heldout/refit arrays):",path)
            continue

        run_probs[path]=np.mean(arrays,axis=0)
        print("added Model4 run:",path,"| variants:",len(arrays))

    except Exception as e:
        print("skip:",path,"|",repr(e)[:160])

print("Runs in optional cross-run test ensemble:",len(run_probs))

if len(run_probs)>=2:
    ens_probs=np.mean(np.stack(list(run_probs.values()),axis=0),axis=0)
    ens_logp=np.log(np.clip(ens_probs,1e-9,1.0))

    teacher_test_pred=pred_from_logits(
        ens_logp,
        "crossrun_model4_ensemble",
    )
    assert validate_test_prediction(teacher_test_pred)

    jp=ROOT/"prediction_crossrun_model4_ensemble.json"
    zp=ROOT/"submission_crossrun_model4_ensemble.zip"

    with open(jp,"w",encoding="utf-8") as f:
        json.dump(teacher_test_pred,f,ensure_ascii=False,indent=2)

    with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as zf:
        zf.write(jp,arcname="prediction.json")

    with zipfile.ZipFile(zp,"r") as zf:
        assert zf.namelist()==["prediction.json"]

    np.savez_compressed(
        ROOT/"crossrun_model4_test_logprobs.npz",
        logp=ens_logp.astype(np.float32),
        doc_id=np.asarray([x[0] for x in test_meta],dtype=object),
        sent_idx=np.asarray([x[1] for x in test_meta],dtype=np.int64),
        domain=np.asarray([x[2] for x in test_meta],dtype=object),
    )
    print("✅",zp.name)
else:
    print("Cross-run ensemble skipped: add at least one explicit external Model4 test_logits.npz path.")


Runs in optional cross-run test ensemble: 1
Cross-run ensemble skipped: add at least one explicit external Model4 test_logits.npz path.


In [26]:
# ============================================================
# CELL 25 — OPTIONAL Model-4 multi-teacher TRAIN soft targets for KD
# ============================================================
# Use this only if you attach teacher-logit files from other strong Model-4 runs
# (for example AdaLoRA). This creates TRAIN-set soft targets for Exp6 KD.
#
# It averages probabilities after exact (doc_id, sent_idx, domain) alignment.
# Test logits are never used here.

EXTRA_MODEL4_TRAIN_TEACHERS = [
    # "/kaggle/input/your-adalora-artifacts/teacher_logits_train_heldout.npz",
]

def load_aligned_train_teacher(path,target_records):
    z=np.load(path,allow_pickle=True)
    required={"logits","doc_id","sent_idx","domain"}
    assert required.issubset(set(z.files)),(path,z.files)

    src_keys=[
        (normalize_id(str(d)),int(s),str(dom))
        for d,s,dom in zip(z["doc_id"],z["sent_idx"],z["domain"])
    ]
    tgt_keys=[
        (normalize_id(r["doc_id"]),int(r["sent_idx"]),str(r["domain"]))
        for r in target_records
    ]

    src_map={k:i for i,k in enumerate(src_keys)}
    assert len(src_map)==len(src_keys),f"duplicate teacher keys: {path}"

    order=[src_map.get(k) for k in tgt_keys]
    assert all(i is not None for i in order),f"teacher rows do not cover train_records: {path}"

    logits=np.asarray(z["logits"],dtype=np.float64)
    assert logits.shape[1]==NUM_LABELS
    return logits[np.asarray(order,dtype=np.int64)]

# Current rsLoRA heldout teacher is always available after Cell 23.
teacher_sources=[
    ("rsLoRA", str(ROOT/"teacher_logits_train_heldout.npz"))
]

for p in EXTRA_MODEL4_TRAIN_TEACHERS:
    teacher_sources.append((Path(p).stem,p))

teacher_probs=[]
used=[]

for name,path in teacher_sources:
    try:
        lg=load_aligned_train_teacher(path,train_records)
        teacher_probs.append(softmax_np(lg))
        used.append(name)
        print("added KD teacher:",name,path)
    except Exception as e:
        print("skip KD teacher:",name,"|",repr(e)[:180])

if len(teacher_probs)>=2:
    avg_prob=np.mean(np.stack(teacher_probs,axis=0),axis=0)
    avg_logp=np.log(np.clip(avg_prob,1e-9,1.0)).astype(np.float32)

    outp=ROOT/"teacher_logits_train_model4_multiteacher.npz"
    np.savez_compressed(
        outp,
        logits=avg_logp,
        y=np.asarray([r["label_id"] for r in train_records],dtype=np.int64),
        doc_id=np.asarray([r["doc_id"] for r in train_records],dtype=object),
        sent_idx=np.asarray([r["sent_idx"] for r in train_records],dtype=np.int64),
        domain=np.asarray([r["domain"] for r in train_records],dtype=object),
        teachers=np.asarray(used,dtype=object),
    )
    print("✅ Saved multi-teacher KD soft targets:",outp)
    print("Teachers:",used)
else:
    print(
        "Multi-teacher KD export skipped: attach at least one additional "
        "Model4 teacher_logits_train_heldout.npz file."
    )


added KD teacher: rsLoRA /kaggle/working/phase3_model4_rslora_teacher/teacher_logits_train_heldout.npz
Multi-teacher KD export skipped: attach at least one additional Model4 teacher_logits_train_heldout.npz file.


## What to keep after the rsLoRA teacher-search run

Download:
- completed notebook
- `best_rslora_rr.pt`
- `final_refit_rslora_rr.pt`
- `history.json`
- `experiment_manifest.json`
- `dev_logits.npz`
- `test_logits.npz`
- `submission_heldout.zip`
- `submission_refit.zip`
- `submission_ensemble.zip`
- `submission_snapshot_ensemble.zip` (if at least 2 top snapshots are available)
- `submission_snapshot_refit_ensemble.zip` (if available)
- `top_snapshots.json`

If rsLoRA becomes the KD teacher, also keep:
- `teacher_logits_train_heldout.npz`
- `teacher_logits_train_dev_refit.npz`

### What to record
- Phase-1 backbone: Model 4
- technique: **Rank-Stabilized LoRA (rsLoRA)**
- rank `r=32`
- alpha `16`
- scaling `alpha/sqrt(r)` = approximately `2.828`
- targets: Q/K/V/O + gate/up/down
- exact trainable parameter count
- best local combined dev Macro-F1 and epoch fraction
- heldout / refit / ensemble Codabench scores

### Run order
1. Attach `eff_full_best*.pt` and the exact `tokenizer.json`; T4 on.
2. Run top-to-bottom.
3. Confirm `strict=True` Phase-1 checkpoint loading succeeds.
4. Confirm **119 rsLoRA modules** (`17 × 7`) and `r=32`.
5. Confirm no unexpected Phase-1 parameter is trainable.
6. Let training select the best checkpoint using 4 dev evaluations/epoch.
7. The exact Phase-3 evaluator must pass before refit.
8. Continue through fractional refit, test inference, structural checks and ZIP creation.
9. Submit `submission_heldout.zip`, `submission_refit.zip`, and `submission_ensemble.zip` if budget allows.
10. Use the highest Codabench score to decide whether rsLoRA or AdaLoRA is the stronger KD teacher.

The final teacher-logit export cell is optional for scoring; run it only if you want the KD artifacts.

- `submission_crossrun_model4_ensemble.zip` is optional and only created if you explicitly list other Model4 `test_logits.npz` files in Cell 24.
- `teacher_logits_train_model4_multiteacher.npz` is optional and only created if you explicitly attach/list another Model4 TRAIN teacher-logit file in Cell 25.
